# **Pengembangan Model MLP: Arsitektur, Regularisasi, dan Training**

Notebook ini membangun arsitektur MLP untuk prediksi risiko diabetes dan menentukan konfigurasinya lewat eksperimen bertahap. Ada tiga hal yang dikerjakan:

1. Merancang arsitektur MLP beserta alasan teknis untuk jumlah layer, neuron, aktivasi, dropout, dan regularisasi L2.
2. Melatih model dengan optimizer Adam, EarlyStopping, dan ModelCheckpoint berdasarkan metrik validation.
3. Membaca kurva loss dan metrik train vs validation untuk melihat tanda overfitting atau underfitting.

Keputusan yang sudah dikunci kelompok sebelum eksperimen:

| Aspek | Keputusan |
|---|---|
| Aktivasi hidden layer | ReLU |
| Optimizer | Adam |
| Input pipeline | `tf.data` lewat helper `make_dataset()` |
| Pengulangan | Setiap konfigurasi dijalankan dengan 3 seed (42, 123, 2026) |
| Loss selama eksperimen | Binary cross-entropy (Focal Loss baru dipasang di notebook fitur unik 1) |
| Data yang dipakai | Train untuk melatih, validation untuk memilih. **Data test tidak disentuh** |

Learning rate dan batch size dituning di `04_eksperimen_optimasi.ipynb`. Di notebook ini keduanya ditahan di nilai awal.

**Penanggung jawab:** Wilbert Edric Tanjiro (231110595)

## **1. Import Library dan Konfigurasi**

Seluruh fungsi model, pipeline data, dan eksperimen ada di `src/model.py`, jadi notebook lain memakai kode yang sama persis.

Eksperimen di notebook ini melatih 30 model dan butuh waktu cukup lama (sekitar 30 sampai 60 menit di CPU, tergantung laptop). Karena itu ada saklar `RUN_EXPERIMENTS`:
- `True`: semua eksperimen dijalankan ulang, lalu hasilnya disimpan ke `reports/results/`.
- `False`: hasil eksperimen yang sudah tersimpan dimuat dari `reports/results/`, sehingga notebook selesai dalam hitungan detik.

In [1]:
import sys
import json
import time
import platform
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
import keras

from src import config
from src import preprocessing as prep
from src import model as M

RUN_EXPERIMENTS = True

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})
WARNA_TRAIN, WARNA_VAL, WARNA_ABU, WARNA_PILIH = "#DD8452", "#4C72B0", "#9A9A9A", "#4C72B0"
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(config.FIGURES_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("Python     :", platform.python_version())
print("Seed       :", M.SEEDS)
print("Konfigurasi awal:", M.DEFAULT_CONFIG)
print("RUN_EXPERIMENTS :", RUN_EXPERIMENTS)

TensorFlow : 2.21.0
Keras      : 3.15.1
Python     : 3.12.10
Seed       : [42, 123, 2026]
Konfigurasi awal: {'hidden_units': [128, 64], 'dropout': 0.2, 'l2': 0.0, 'batch_norm': False, 'learning_rate': 0.001, 'batch_size': 512}
RUN_EXPERIMENTS : True


## **2. Memuat Data Hasil Preprocessing**

Data diambil lewat `prepare_datasets()` dari `src/preprocessing.py`. Fingerprint data test dicocokkan dengan yang tersimpan di `models/preprocessing_config.json` untuk memastikan pembagian data sama dengan notebook 02. Data test ikut dimuat karena berasal dari fungsi yang sama, tetapi tidak dipakai sama sekali di notebook ini.

In [2]:
data = prep.prepare_datasets()
cfg_prep = prep.load_preprocessing_config()
assert data["test_fingerprint"] == cfg_prep["split"]["test_fingerprint"], "Split data berbeda dengan notebook 02"

for part in ["train", "val"]:
    X, y = data[f"X_{part}"], data[f"y_{part}"]
    print(f"{part:<5}: X {X.shape}, {X.dtype} | kelas 1 = {y.mean():.2%}")
print("Fingerprint test cocok:", data["test_fingerprint"])

train: X (160211, 21), float32 | kelas 1 = 15.30%
val  : X (34332, 21), float32 | kelas 1 = 15.30%
Fingerprint test cocok: 01b58633cb6f22b5
